# P4 RAG の評価 ― 検索と回答を分けて測る

> **この教材に登場する会社・人物・製品・数値・文書はすべて架空のものです。実在の企業・団体・個人とは関係ありません。**

ヒイラギ運輸（架空）の社内規程 14 本と質問 30 問で、検索（BM25）と回答（Opus 5.5 / Sonnet 5.5）を分けて測ります。要素の照合・断る率・忠実さの判定（判定役は回答者と別のモデル）と、照合そのものの誤りを読みます。

- 上から順に実行します。0 のセルは準備です（Colab でも手元の Jupyter でも同じ）。
- API キーが無ければ講師の記録の再生（replay）で、本物の実行と同じ数字が出ます（費用 0）。
- 🔴 データはすべて架空の会社・文書です。

## 0. 準備

In [ ]:
# 0-1. 実行場所を見分けて、キットのフォルダに移る（Colab のときだけ取得とインストールをする）
import os, sys, subprocess
REPO_URL = "https://github.com/gucchi123/llm-eval-colab.git"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    KIT = "/content/llm-eval-kit"
    if not os.path.exists(KIT):
        subprocess.run(["git", "clone", "-q", REPO_URL, KIT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)
else:  # 手元: notebooks/ から開いた前提。キットの根（common/ がある所）を探す
    KIT = os.path.abspath(".." if os.path.exists("../common/llm.py") else ".")
os.chdir(KIT)
sys.path.insert(0, KIT)
print("実行場所:", "Colab" if IN_COLAB else "手元の Jupyter", "/ キットの根:", os.path.basename(KIT))

In [ ]:
# 0-2. AI の呼び先を決める（キーは表示しない）
from common import nb
from common.nb import choose_provider, safe, show, show_file, lab
PROVIDER = choose_provider(IN_COLAB)

## 1. 検索の指標のコード ― 再現率・逆順位・割引累積利得
`common/metrics.py` の本物のコードです。

In [ ]:
from common import kit, llm as L, metrics as M, stats
show(M.recall_at_k)
show(M.mrr)
show(M.ndcg_at_k)

## 2. 一つの問いで検索を測り、30 問の平均を出す
根拠の規程が 2 本ある問い Q22 の、BM25 の上位 5 件。

In [ ]:
rr = lab("run_rag", "lab04_rag_eval")
pol = kit.load_json("policies.json")
docs, qs = pol["docs"], {q["id"]: q for q in pol["questions"]}
bm = rr.BM25(docs)
q = qs["Q22"]
ranked = [i for i, _ in bm.search(q["question"], 5)]
print(q["question"]); print("上位 5 件:", ranked, "/ 正解の根拠:", q["gold_docs"])
gold = set(q["gold_docs"])
print(f"recall@1 {M.recall_at_k(ranked, gold, 1):.2f} / recall@3 {M.recall_at_k(ranked, gold, 3):.2f}"
      f" / 逆順位 {M.mrr(ranked, gold):.2f} / nDCG@3 {M.ndcg_at_k(ranked, gold, 3):.3f}")
rk = [x for x in pol["questions"] if x["gold_docs"]]
avg = lambda f: sum(f([i for i, _ in bm.search(x["question"], 14)], set(x["gold_docs"])) for x in rk) / len(rk)
print(f"根拠のある {len(rk)} 問の平均: recall@3 {avg(lambda r, g: M.recall_at_k(r, g, 3)):.3f} / 逆順位 {avg(M.mrr):.3f}"
      f" / nDCG@3 {avg(lambda r, g: M.ndcg_at_k(r, g, 3)):.3f}")

## 3. 要素の照合と、断ったかどうかの判定（照合 v1）
答えに入るべき語が全部あるか。「規程に記載がありません」で始まれば断ったとみなす…はずでした。

In [ ]:
show(rr.check_answer)
print("断りの文:", pol["abstain"])
print("Q11 で入るべき語:", qs["Q11"]["must"])

## 4. 30 問を回答させ、照合 v1 で採点する（回答 Opus 5.5）
ラボの本物の実行（`run_rag.py`）の出力。長い回答の行は省きます。

In [ ]:
import io, contextlib
P = "replay" if PROVIDER == "mock" else PROVIDER
def quiet(argv):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        rr.main(["--provider", P, "--all"] + argv)
    return buf.getvalue().splitlines()
with safe():
    for line in quiet(["--model", "claude-opus-5-5"]):
        if line.strip() and not line.startswith(("   ×", "   20", "検索", "   上位")):
            print(line[:96])

## 5. 根拠を抜いたら、断れるか（断る率）
検索結果から正解の根拠の規程を抜いた 26 問。根拠が無いのに数字を作ったら失格です。

In [ ]:
with safe():
    for m in ("claude-opus-5-5", "claude-sonnet-5-5"):
        out = quiet(["--model", m, "--condition", "degraded"])
        runs = kit.read_jsonl(f"lab04_rag_eval/out/runs_rag_degraded_{kit.slug(m)}.jsonl")
        ans = [r for r in runs if not r["abstained"]]
        print(f"{m:<18} 根拠を抜いた {len(runs)} 問 → 断った {len(runs) - len(ans)} 問 / 断らずに答えた {len(ans)} 問")
        print("   例", runs[0]["id"], "の回答:", repr(runs[0]["answer"][:40]))

## 6. 忠実さの判定 ― 判定役は回答者と別のモデル
回答を主張に分け、渡した規程から導けるかを判定します（依頼文 `prompts/faithfulness.md`）。

In [ ]:
print("# lab04_rag_eval/prompts/faithfulness.md（長い行は折り返して表示）")
for n, line in enumerate(open("lab04_rag_eval/prompts/faithfulness.md", encoding="utf-8").read().splitlines(), 1):
    cut = 44 if sum(ord(ch) > 0x2E80 for ch in line) > 20 else 200
    print(f"{n:4d}  {line[:cut]}")
    if len(line) > cut:
        print("      " + line[cut:])

## 7. 忠実さを測る（回答 Opus 5.5・判定役 Sonnet 5.5）
断らなかった回答だけを判定します。

In [ ]:
with safe():
    out = quiet(["--model", "claude-opus-5-5", "--judge"])
    runs = kit.read_jsonl("lab04_rag_eval/out/runs_rag_normal_opus-5-5.jsonl")
    f = [r for r in runs if r.get("faithfulness") is not None]
    print(f"判定した回答 {len(f)} 問 / 忠実さの平均 {sum(r['faithfulness'] for r in f) / len(f):.3f}")
    print(f"主張の数 {sum(len(r['claims']) for r in f)} / 導けないとされた主張 "
          f"{sum(1 for r in f for c in r['claims'] if not c.get('supported'))}")
    for r in f:
        if r["faithfulness"] < 1:
            print(f"   {r['id']}: 忠実さ {r['faithfulness']:.2f}（主張 {len(r['claims'])} 個）")
    print([line for line in out if line.startswith("費用")][0])

## 8. 照合の誤りを読む ― × を全件読む
照合 v1 で × になった問いを、v2（断りの判定と別表記を直したもの）と並べます。

In [ ]:
import inspect
src, first = inspect.getsourcelines(rr.check_answer_v2)
print("# lab04_rag_eval/run_rag.py", first, "行目から（長い行は折り返して表示）")
for n, line in enumerate(src, first):
    line = line.rstrip()
    print(f"{n:4d}  {line[:88]}")
    if len(line) > 88:
        print("      " + " " * 8 + line[88:])
print("別表記:", rr.EXTRA_ALIASES)

## 9. 照合 v1 と v2 を並べる ― 外れの多くは照合の誤り
2 つの回答役の、v1 で × になった問い。

In [ ]:
for m in ("claude-opus-5-5", "claude-sonnet-5-5"):
    runs = kit.read_jsonl(f"lab04_rag_eval/out/runs_rag_normal_{kit.slug(m)}.jsonl")
    print(f"{m}: v1 {sum(r['correct'] for r in runs)}/30 → v2 {sum(r['correct_v2'] for r in runs)}/30")
    for r in runs:
        if not r["correct"]:
            head = " ".join(r["answer"].replace("*", "").replace("#", "").split())[:26]
            print(f"   {r['id']} v1 × → v2 {'○' if r['correct_v2'] else '×'}  回答の頭:「{head}」")

## 10. 忠実さの判定も間違える ― Q13
「導けない」とされた主張と、根拠の規程を並べます。

In [ ]:
r13 = next(r for r in kit.read_jsonl("lab04_rag_eval/out/runs_rag_normal_opus-5-5.jsonl") if r["id"] == "Q13")
print("問い:", qs["Q13"]["question"])
for c in r13["claims"]:
    if not c.get("supported"):
        print("導けないとされた主張:", c["claim"])
d10 = next(d for d in docs if d["id"] == "D10")
print(f"根拠の規程 [{d10['id']}] {d10['title']}:")
t = d10["text"]
while t:
    print("   ", t[:44]); t = t[44:]

## 11. 例題 ― 三問で上位三件の再現率と逆順位を手で出す
BM25 の上位 3 件と正解の根拠を見比べ、手で数えてから metrics の答えと比べます。

In [ ]:
for qid in ("Q01", "Q10", "Q22"):
    q = qs[qid]
    top = [i for i, _ in bm.search(q["question"], 14)]
    gold = set(q["gold_docs"])
    hit = [i for i in top[:3] if i in gold]
    first = next(n for n, i in enumerate(top, 1) if i in gold)
    print(f"{qid} 上位3 {top[:3]} 正解 {sorted(gold)} → 再現率 {len(hit)}/{len(gold)}"
          f" / 最初の正解は {first} 位 → 逆順位 1/{first}={1 / first:.3f}")
    print(f"     metrics: recall@3 {M.recall_at_k(top, gold, 3):.3f} / MRR {M.mrr(top, gold):.3f}")